# Comparaison : prix fixe vs Q-table (Monte-Carlo) vs DQN

Chaque agent affronte **la même série de scénarios de randomisation de domaine** (mêmes graines).
Comparaison **appariée** : les écarts ne viennent pas de la chance du tirage d'artiste / de salle.

Deux prix fixes de référence :
- **`fixed` = 70 CHF** : ce qu'un organisateur choisit *réellement* (instinct : remplir la salle) ;
- **`fixed_90` = 90 CHF** : l'optimum empirique « oracle », inconnaissable a priori.

Deux axes d'évaluation (cahier des charges) : **chiffre d'affaires** (`ca` = billetterie + revenus
annexes) et **taux de remplissage**. `revenue` = marge de contribution (l'objectif que les agents RL optimisent).

In [ ]:
# Recharge automatiquement les modules du projet quand leur code change
# (évite de redémarrer le kernel après une modif de env/ ou pricing_agent/)
%load_ext autoreload
%autoreload 2

import os
import sys

sys.path.append(os.path.abspath('../env'))
sys.path.append(os.path.abspath('../pricing_agent'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from gym_wrapper import TicketEnv
from agent_fixed import FixedPriceAgent
from agent_qtable import QTableAgent
from evaluate import compare_agents, summarize

HORIZON = 120
N_EPISODES = 300

In [ ]:
# Recharge la chaîne complète DANS L'ORDRE des dépendances (au cas où le kernel a une version périmée).
# Noms en clair -> pas de collision avec la variable `agents` (liste) définie plus bas.
import importlib
for _name in ("refdata", "agents", "market", "gym_wrapper",
              "registry", "agent_fixed", "agent_qtable", "agent_dqn", "evaluate"):
    importlib.reload(importlib.import_module(_name))

# rebind APRÈS reload (sinon les noms pointent encore sur l'ancienne version)
from gym_wrapper import TicketEnv
from agent_fixed import FixedPriceAgent
from agent_qtable import QTableAgent
from evaluate import compare_agents, summarize

# Garde-fou : si l'env n'a pas les champs récents, le reload a échoué -> redémarrer le kernel.
if "ca_total" not in TicketEnv(horizon=HORIZON).reset(seed=0)[1]:
    raise RuntimeError("Environnement périmé — fais 'Kernel > Restart Kernel' puis Run All.")

qtable = QTableAgent.load()
print("Q-table :", qtable.path.name, "|", qtable.meta.get("created", ""), "|", qtable.meta.get("method", ""))

agents = [
    FixedPriceAgent.from_price(70),        # prix fixe "réaliste" : instinct de remplir la salle
    FixedPriceAgent.from_price(90),        # prix fixe optimum "oracle" (inconnaissable a priori)
    qtable,
]
agents[0].name = "fixed"
agents[1].name = "fixed_90"

try:                                    # DQN si TensorFlow est installé et un modèle existe
    from agent_dqn import DQNAgent
    dqn = DQNAgent.load()
    print("DQN     :", dqn.path.name, "|", dqn.meta.get("created", ""))
    agents.append(dqn)
except (ImportError, FileNotFoundError) as e:
    print("DQN ignoré :", e)

results = compare_agents(agents, lambda: TicketEnv(horizon=HORIZON), n_episodes=N_EPISODES)
summarize(results)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

order = results.groupby('agent')['revenue'].mean().sort_values().index

results.boxplot(column='revenue', by='agent', ax=axes[0])
axes[0].set_title('Chiffre d\'affaires par épisode'); axes[0].set_xlabel(''); axes[0].set_ylabel('CHF')

results.boxplot(column='fill_rate', by='agent', ax=axes[1])
axes[1].set_title('Taux de remplissage'); axes[1].set_xlabel(''); axes[1].set_ylabel('fraction')

for name in order:
    sub = results[results.agent == name].sort_values('popularity')
    axes[2].scatter(sub['popularity'], sub['revenue'], s=10, alpha=0.4, label=name)
axes[2].set_title('CA vs popularité de l\'artiste'); axes[2].set_xlabel('indice de popularité'); axes[2].set_ylabel('CHF')
axes[2].legend()

plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# --- Comparaison appariée (même graine = même scénario) --------------------
wide_ca = results.pivot_table(index="seed", columns="agent", values="ca")

print("CA moyen vs prix fixe RÉALISTE (70 CHF) :")
for a in [c for c in wide_ca.columns if c != "fixed"]:
    g = (wide_ca[a] - wide_ca["fixed"]).mean() / wide_ca["fixed"].mean() * 100
    print(f"  {a:10s} {g:+6.1f} %   (bat le prix fixe dans {(wide_ca[a] > wide_ca['fixed']).mean():.0%} des scénarios)")

if "fixed_90" in wide_ca.columns:
    print("\nCA moyen vs prix fixe OPTIMUM (90 CHF, oracle) :")
    for a in [c for c in wide_ca.columns if c not in ("fixed", "fixed_90")]:
        g = (wide_ca[a] - wide_ca["fixed_90"]).mean() / wide_ca["fixed_90"].mean() * 100
        print(f"  {a:10s} {g:+6.1f} %   (bat l'optimum dans {(wide_ca[a] > wide_ca['fixed_90']).mean():.0%} des scénarios)")

print("\nRemplissage moyen :")
print(results.groupby("agent")["fill_rate"].mean().round(3).sort_values(ascending=False).to_string())